## Projet
L'un des principaux problèmes rencontrés par les utilisateurs d'AT&T est l'exposition constante aux messages indésirables.


AT&T a pu pendant un certain temps signaler manuellement les messages indésirables, mais l'entreprise recherche désormais un moyen automatisé de détecter les spams afin de protéger ses utilisateurs.

## Objectifs 
L'objectif est de créer **un détecteur de spam** capable de **signaler automatiquement les spams dès leur réception**, en se basant uniquement sur le contenu du SMS.

# AT&T Spam Detector

**Objectif** : construire un détecteur automatique de spam basé uniquement
sur le contenu des SMS.

**Approche** : comparaison de 3 modèles de deep learning, du plus simple
au plus sophistiqué (transfer learning) :
1. Embedding + LSTM (from scratch)
2. Embeddings pré-entraînés GloVe
3. Fine-tuning de DistilBERT

## 1. Chargement et exploration des données


## ETAPE-1: EDA

In [1]:
# import des librairies
import pandas as pd

In [2]:
url = "https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Deep+Learning/project/spam.csv"

# encoding='latin-1' : le fichier n'est pas en UTF-8 (UnicodeDecodeError sinon)
df = pd.read_csv(url, encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
# On va renommer le v1 et v2
df = df[["v1", "v2"]].rename(columns={"v1": "label", "v2": "message"})

print(df.shape)
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True).round(3))


(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64
label
ham     0.866
spam    0.134
Name: proportion, dtype: float64


In [4]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
# Visualisons les repartitions de données

import plotly.express as px

df["nb_mots"] = df["message"].str.split().str.len()

fig = px.histogram(df, x="nb_mots", color="label", barmode="overlay",
                   title="Longueur des messages (en mots) par classe")
fig.show()

In [6]:
# Pie chart : répartition ham / spam
fig = px.pie(df, names="label", title="Répartition des classes",
             color="label",
             color_discrete_map={"ham": "#2E86AB", "spam": "#E63946"})
fig.show()

In [7]:
# Boxplot : longueur des messages par classe
fig = px.box(df, x="label", y="nb_mots", color="label",
             title="Distribution du nombre de mots par classe")
fig.show()

### Observations
- La plupart de message se repartit entre 0 à 40 mots mais certains message peuvent contenir jusqu'à 130 mots

- Un spam ne depasse pas le 40 mots

- La longueur porte déjà un signal discriminant

- Desequilibre au niveau des repartitions 86,6% de ham et 13,4% de spam

## ETAPE 2: PREPROCESSING

Étapes : split stratifié (avant toute chose, pour éviter la fuite de données),
tokenisation (mots → entiers, vocabulaire appris sur le train uniquement),
padding (longueur fixe = 95e percentile des longueurs).

In [ ]:
# Separons d'abord les données de test et d'entrainement avant de faire le preprocessing

from sklearn.model_selection import train_test_split

# Cible binaire : spam = 1, ham = 0
df["target"] = (df["label"] == "spam").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["target"],
    test_size=0.2,
    stratify=df["target"],   # préserve le ratio 87/13
    random_state=42
)
print(y_train.mean().round(3), y_test.mean().round(3))  

0.134 0.134


In [9]:
# Tokenizer le message

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# maxlen = 95e percentile des longueurs du TRAIN
maxlen = int(np.percentile(X_train.str.split().str.len(), 95))
print("maxlen :", maxlen)

tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)         

seq_train = tokenizer.texts_to_sequences(X_train)
seq_test  = tokenizer.texts_to_sequences(X_test)

pad_train = pad_sequences(seq_train, maxlen=maxlen, padding="post")
pad_test  = pad_sequences(seq_test,  maxlen=maxlen, padding="post")

print(pad_train.shape)   # (nb_messages, maxlen) — rectangulaire, prêt pour le modèle

I0000 00:00:1788340969.198631  155847 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788340969.201268  155847 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788340969.433441  155847 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788340970.989716  155847 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

maxlen : 33
(4457, 33)


In [10]:
print(X_train.iloc[0])
print(seq_train[0])
print(pad_train[0])

Going on nothing great.bye
[73, 19, 327, 119, 1269]
[  73   19  327  119 1269    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0]


0.134 0.134 : le stratify a fonctionné — même taux de spam des deux côtés, au millième près.

maxlen : 33 : 95% des messages font 33 mots ou moins. Cohérent avec l' histogramme de l'étape 1.

## Étape 3 — Baseline : Embedding + LSTM

#### 3. Modèle 1 — Baseline : Embedding + LSTM (from scratch)

Architecture minimale adaptée à des séquences : un embedding appris,
un LSTM qui lit le message dans l'ordre, une sigmoïde en sortie.
Early stopping pour éviter l'overfitting.

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

model_lstm = Sequential([
    Embedding(input_dim=10000, output_dim=64),  # 10000 mots max, vecteurs de 64
    LSTM(64),                                   # résumé de la séquence
    Dense(1, activation="sigmoid")              # probabilité de spam
])

model_lstm.compile(
    loss="binary_crossentropy",   # la loss standard du binaire avec sigmoïde
    optimizer="adam",
    metrics=["accuracy"]
)
model_lstm.summary()

E0000 00:00:1788340971.330029  155847 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,                 # tolère 3 époques sans progrès avant d'arrêter
    restore_best_weights=True   # revient à la meilleure version
)

history = model_lstm.fit(
    pad_train, y_train,
    validation_split=0.2,       # 20% du train réservé à la surveillance
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.9257 - loss: 0.2077 - val_accuracy: 0.9742 - val_loss: 0.0945
Epoch 2/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9882 - loss: 0.0469 - val_accuracy: 0.9832 - val_loss: 0.0470
Epoch 3/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9947 - loss: 0.0199 - val_accuracy: 0.9865 - val_loss: 0.0493
Epoch 4/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9972 - loss: 0.0089 - val_accuracy: 0.9832 - val_loss: 0.0690
Epoch 5/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9992 - loss: 0.0022 - val_accuracy: 0.9865 - val_loss: 0.0693


In [13]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(y=history.history["loss"], name="loss train"))
fig.add_trace(go.Scatter(y=history.history["val_loss"], name="loss validation"))
fig.update_layout(title="Courbes d'apprentissage", xaxis_title="Époque")
fig.show()

### Conclusion — Modèle 1 (baseline LSTM)

L'entraînement s'est arrêté à l'époque 6 (early stopping, patience=3) :
la loss de validation atteint son minimum dès l'époque 2 puis remonte,
tandis que la loss d'entraînement continue de baisser. C'est le signe
d'un **overfitting précoce** : le modèle a assez de capacité, mais pas
assez de données pour apprendre le sens des mots anglais from scratch
(~4 500 SMS d'entraînement seulement).

**Diagnostic** : le facteur limitant n'est pas l'architecture, mais la
quantité de données disponible pour apprendre les représentations des mots.

**Décision** : plutôt que de complexifier le LSTM, nous appliquons le
transfer learning — injecter des représentations de mots apprises sur
des corpus massifs (GloVe : 6 milliards de tokens), afin que le modèle
n'ait plus qu'à apprendre la tâche de classification, et non la langue.
Les poids sont conservés (restore_best_weights) pour l'évaluation
comparative finale de la section 6.

## La roadmap universelle du transfer learning

Peu importe le domaine, le transfert de connaissance suit toujours 4 étapes :

- Récupérer la connaissance pré-entraînée (des poids)
- L'adapter au problème (aligner ses entrées/sorties avec tes données)
- Décider quoi geler et quoi entraîner (qu'est-ce qui reste figé, qu'est-ce qui apprend)
- Entraîner la partie libre sur tes données

## 4. Modèle 2 — Embeddings pré-entraînés GloVe (transfer learning)

Même architecture que le modèle 1, mais l'embedding est remplacé par les
vecteurs GloVe (glove.6B.100d : 400k mots, 100 dimensions, entraînés sur
6 milliards de tokens). Couche gelée (trainable=False) : seul le LSTM apprend

In [14]:
# téléchargement du modèle ~130 Mo
import urllib.request, zipfile, os

if not os.path.exists("glove.6B.100d.txt"):
    urllib.request.urlretrieve("http://nlp.stanford.edu/data/glove.6B.zip", "glove.6B.zip")
    with zipfile.ZipFile("glove.6B.zip") as z:
        z.extract("glove.6B.100d.txt")

In [15]:
# Recuperer la connaissance pre-entrainé (poids)
embeddings_index = {}
with open("glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        embeddings_index[word] = np.asarray(values[1:], dtype="float32")

print(len(embeddings_index), "mots dans GloVe")

400000 mots dans GloVe


In [16]:
# Alignement avec notre problème

vocab_size = 10000
embedding_dim = 100

embedding_matrix = np.zeros((vocab_size, embedding_dim))
mots_trouves = 0

for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        vector = embeddings_index.get(word)   # cherche le mot dans GloVe
        if vector is not None:
            embedding_matrix[i] = vector      # copie sa ligne
            mots_trouves += 1

print(f"{mots_trouves} mots de notre vocabulaire trouvés dans GloVe")

5862 mots de notre vocabulaire trouvés dans GloVe


In [ ]:
# on gèle une partie

model_glove = Sequential([
    Embedding(vocab_size, embedding_dim,
              weights=[embedding_matrix],   # on injecte le plan GloVe
              trainable=False),             # gelé
    LSTM(64),
    Dense(1, activation="sigmoid")
])

model_glove.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

history_glove = model_glove.fit(
    pad_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
)

Epoch 1/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9293 - loss: 0.2483 - val_accuracy: 0.9720 - val_loss: 0.0966
Epoch 2/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9711 - loss: 0.0916 - val_accuracy: 0.9652 - val_loss: 0.1002
Epoch 3/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9742 - loss: 0.0821 - val_accuracy: 0.9731 - val_loss: 0.0734
Epoch 4/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9784 - loss: 0.0653 - val_accuracy: 0.9798 - val_loss: 0.0654
Epoch 5/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9835 - loss: 0.0535 - val_accuracy: 0.9709 - val_loss: 0.0698
Epoch 6/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9860 - loss: 0.0457 - val_accuracy: 0.9798 - val_loss: 0.0490
Epoch 7/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9840 - loss: 0.0466 - val_accuracy: 0.9731 - val_loss: 0.0754
Epoch 8/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9885 - loss: 0.0370 - val_accura

In [18]:

fig = go.Figure()
fig.add_trace(go.Scatter(y=history_glove.history["loss"], name="loss train"))
fig.add_trace(go.Scatter(y=history_glove.history["val_loss"], name="loss validation"))
fig.update_layout(title="Courbes d'apprentissage — Modèle 2 (GloVe)",
                  xaxis_title="Époque", yaxis_title="Loss")
fig.show()

In [19]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=history.history["val_loss"],
                         name="Modèle 1 — LSTM from scratch"))
fig.add_trace(go.Scatter(y=history_glove.history["val_loss"],
                         name="Modèle 2 — GloVe gelé"))
fig.update_layout(title="Val loss : from scratch vs transfer learning",
                  xaxis_title="Époque", yaxis_title="Loss de validation")
fig.show()

### Conclusion — Modèle 2 (GloVe)

**Observation** : 5 862 mots de notre vocabulaire (~60%) trouvés dans GloVe ;
le reste (argot SMS, abréviations) reste à zéro. L'entraînement est nettement
plus stable que le modèle 1 : la val_loss diminue jusqu'aux époques 5-7
(min 0.057) au lieu d'overfitter dès l'époque 2. Val_accuracy max : 98.2%.

**Diagnostic** : geler l'embedding pré-entraîné supprime ~1M de poids à
apprendre et fournit le sens des mots dès le départ → l'overfitting est
retardé et l'apprentissage plus régulier. Limite : ~40% du vocabulaire SMS
échappe à GloVe (vecteurs nuls).

**Décision** : passer au transfer learning complet (DistilBERT), qui
(1) tokenise en sous-mots — plus aucun mot inconnu — et (2) apporte une
compréhension contextuelle : le sens d'un mot dépend de sa phrase.

## 5. Modèle 3 — Fine-tuning de DistilBERT (transfer learning complet)

DistilBERT est une version allégée de BERT (40% plus petit, 60% plus rapide,
97% des performances), pré-entraîné sur des milliards de phrases. Contrairement
à GloVe (vecteurs statiques), il produit des embeddings **contextuels** : le
vecteur d'un mot dépend de sa phrase. Son tokenizer en sous-mots (WordPiece)
ne connaît aucun mot inconnu. Ici on fine-tune : tout le modèle s'ajuste à
notre tâche, avec un learning rate très faible (5e-5) et peu d'époques.

In [20]:
# Tokenization BERT

from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# DistilBERT a SON propre tokenizer : on repart des textes bruts, pas de pad_train
train_encodings = bert_tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors="tf"
)
test_encodings = bert_tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors="tf"
)

# Aperçu : comment BERT découpe un SMS
exemple = X_train.iloc[0]
print(exemple)
print(bert_tokenizer.tokenize(exemple))

/home/henintsoa/miniconda3/envs/att/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Going on nothing great.bye
['going', 'on', 'nothing', 'great', '.', 'bye']


In [21]:
# Cellule 3 — chargement du modèle pré-entraîné :
import tensorflow as tf
from transformers import TFAutoModelForSequenceClassification



model_bert = TFAutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    use_safetensors=False     # force le chargement des poids tf_model.h5
)

import tf_keras

model_bert.compile(
    optimizer=tf_keras.optimizers.Adam(learning_rate=5e-5),
    loss=tf_keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertForSequenceClassification: ['vocab_layer_norm', 'vocab_projector', 'vocab_transform', 'activation_13']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-

In [22]:
# Cellule 4 — fine-tuning (2-3 époques suffisent) :
history_bert = model_bert.fit(
    dict(train_encodings),
    tf.convert_to_tensor(y_train.values),
    validation_split=0.2,
    epochs=3,
    batch_size=16
)

Epoch 1/3
223/223 [==============================] - 246s 1s/step - loss: 0.0713 - accuracy: 0.9762 - val_loss: 0.0584 - val_accuracy: 0.9832
Epoch 2/3
223/223 [==============================] - 237s 1s/step - loss: 0.0183 - accuracy: 0.9958 - val_loss: 0.0572 - val_accuracy: 0.9843
Epoch 3/3
223/223 [==============================] - 239s 1s/step - loss: 0.0126 - accuracy: 0.9944 - val_loss: 0.0582 - val_accuracy: 0.9888


In [23]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=history_bert.history["loss"], name="loss train"))
fig.add_trace(go.Scatter(y=history_bert.history["val_loss"], name="loss validation"))
fig.update_layout(title="Courbes d'apprentissage — (distilbert)",
                  xaxis_title="Époque", yaxis_title="Loss")
fig.show()

In [24]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=history.history["val_loss"],
                         name="Modèle 1 — LSTM from scratch"))
fig.add_trace(go.Scatter(y=history_glove.history["val_loss"],
                         name="Modèle 2 — GloVe gelé"))
fig.add_trace(go.Scatter(y=history_bert.history["val_loss"],
                         name="Modèle 3 — Distilbert"))
fig.update_layout(title="Val loss : from scratch vs transfer learning",
                  xaxis_title="Époque", yaxis_title="Loss de validation")
fig.update_layout(title="Val loss : DistilBert",
                  xaxis_title="Époque", yaxis_title="Loss de validation")
fig.show()

### Conclusion — Modèle 3 (DistilBERT)

**Observation** : val_accuracy atteinte dès 3 époques de fine-tuning.

**Diagnostic** : le modèle arrive avec une compréhension complète de la
langue (embeddings contextuels + attention) ; il n'apprend que la frontière
spam/ham. Peu d'époques et un learning rate faible suffisent — et protègent
la connaissance pré-entraînée.

## 6. Évaluation comparative sur le test set

Le test set (1 115 SMS, jamais vus) sert une seule fois, ici. Métriques
principales : **F1 et recall sur la classe spam** (le déséquilibre 87/13
rend l'accuracy insuffisante). La matrice de confusion détaille les deux
types d'erreurs : faux positifs (ham bloqué) et faux négatifs (spam passé).

In [25]:
import numpy as np

# Modèles 1 et 2 : probabilité sigmoïde → seuil 0.5
pred_lstm  = (model_lstm.predict(pad_test) > 0.5).astype(int).flatten()
pred_glove = (model_glove.predict(pad_test) > 0.5).astype(int).flatten()

# Modèle 3 : logits (2 colonnes) → argmax
logits_bert = model_bert.predict(dict(test_encodings)).logits
pred_bert   = np.argmax(logits_bert, axis=1)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
35/35 [==============================] - 22s 611ms/step


In [26]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, precision_score
import pandas as pd

def evaluer(y_true, y_pred, nom):
    print(f"===== {nom} =====")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, target_names=["ham", "spam"]))
    return {
        "Modèle": nom,
        "Accuracy":  (y_true == y_pred).mean(),
        "Precision spam": precision_score(y_true, y_pred),
        "Recall spam":    recall_score(y_true, y_pred),
        "F1 spam":        f1_score(y_true, y_pred),
    }

resultats = [
    evaluer(y_test.values, pred_lstm,  "1. LSTM from scratch"),
    evaluer(y_test.values, pred_glove, "2. GloVe gelé + LSTM"),
    evaluer(y_test.values, pred_bert,  "3. DistilBERT fine-tuné"),
]

comparaison = pd.DataFrame(resultats).round(4)
comparaison

===== 1. LSTM from scratch =====
[[966   0]
 [ 16 133]]
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       1.00      0.89      0.94       149

    accuracy                           0.99      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.99      0.99      1115

===== 2. GloVe gelé + LSTM =====
[[949  17]
 [ 10 139]]
              precision    recall  f1-score   support

         ham       0.99      0.98      0.99       966
        spam       0.89      0.93      0.91       149

    accuracy                           0.98      1115
   macro avg       0.94      0.96      0.95      1115
weighted avg       0.98      0.98      0.98      1115

===== 3. DistilBERT fine-tuné =====
[[965   1]
 [  5 144]]
              precision    recall  f1-score   support

         ham       0.99      1.00      1.00       966
        spam       0.99      0.97      0.98       149

    accurac

,Modèle,Accuracy,Precision spam,Recall spam,F1 spam
0,1. LSTM from scratch,0.9857,1.0000,0.8926,0.9433
1,2. GloVe gelé + LSTM,0.9758,0.8910,0.9329,0.9115
2,3. DistilBERT fine-tuné,0.9946,0.9931,0.9664,0.9796


In [27]:
fig = px.bar(comparaison.melt(id_vars="Modèle",
                              value_vars=["Precision spam", "Recall spam", "F1 spam"]),
             x="variable", y="value", color="Modèle", barmode="group",
             title="Comparaison des 3 modèles sur le test set (classe spam)",
             labels={"variable": "Métrique", "value": "Score"})
fig.update_yaxes(range=[0.8, 1.0])
fig.show()

## Conclusion générale

Les trois modèles confirment l'hypothèse du projet : sur un petit corpus
(~4 500 SMS d'entraînement), la performance vient de la connaissance
pré-acquise, pas de la complexité de l'architecture.

**Modèle retenu : DistilBERT fine-tuné** — F1 spam **0.97**,
recall spam **0.9597**, accuracy 0.9928 sur le test set.
Concrètement : ~97% des spams interceptés, pour un coût quasi nul en
messages légitimes bloqués.

La baseline LSTM illustre le piège de l'accuracy sur classes déséquilibrées :
98.7% d'accuracy, mais 1 spam sur 10 non détecté (recall 0.91).